In [1]:
from tensorflow.keras.preprocessing.image import img_to_array, load_img
import numpy as np
import glob
import os 
import cv2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv3D, ConvLSTM2D, Conv3DTranspose 
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
import imutils

I0000 00:00:1791320984.333986   95144 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
store_image = []
train_path = './train/UCSD_Anomaly_Dataset.v1p2/UCSDped1/Train'
fps = 5
train_videos = sorted(os.listdir(train_path))
train_images_path = train_path + '/frames'
os.makedirs(train_images_path, exist_ok=True)

In [3]:
def store_inarray(image_path):
    image = cv2.imread(image_path)                      # BGR, uint8
    image = cv2.resize(image, (227,227), interpolation=cv2.INTER_AREA)
    gray = 0.1140 * image[:,:,0] + 0.5870 * image[:,:,1] + 0.2989 * image[:,:,2]  # BGR order
    store_image.append(gray)

In [4]:
for video in train_videos:
    video_dir = os.path.join(train_path, video)
    if not os.path.isdir(video_dir):
        continue
    for image in sorted(os.listdir(video_dir)):
        if not image.endswith('.tif'):
            continue
        image_path = os.path.join(video_dir, image)
        store_inarray(image_path)

In [5]:
store_image = np.array(store_image, dtype=np.float32)   # (N, H, W)
print(store_image.shape)                                # must print (N, 227, 227)
store_image = np.transpose(store_image, (1, 2, 0))      # (H, W, N)
store_image = (store_image - store_image.min()) / (store_image.max() - store_image.min())
np.save('training.npy', store_image)     

(6800, 227, 227)


In [6]:
stae_model = Sequential()
stae_model.add(Conv3D(filters=128, kernel_size=(11,11,1), strides=(4,4,1), padding='valid', input_shape=(227,227, 10,1), activation='tanh'))
stae_model.add(Conv3D(filters=64, kernel_size=(5,5,1), strides=(2,2,1), padding='valid', activation='tanh'))
stae_model.add(ConvLSTM2D(filters=64, kernel_size=(3,3), strides=(1,1), padding='same', dropout=0.4, recurrent_dropout=0.3, return_sequences=True))
stae_model.add(ConvLSTM2D(filters=32, kernel_size=(3,3), strides=(1,1), padding='same', dropout=0.3, return_sequences=True))
stae_model.add(ConvLSTM2D(filters=64, kernel_size=(3,3), strides=(1,1), padding='same', dropout=0.5, return_sequences=True))
stae_model.add(Conv3DTranspose(filters=128, kernel_size=(5,5,1), strides=(2,2,1), padding='valid', activation='tanh'))
stae_model.add(Conv3DTranspose(filters=1, kernel_size=(11,11,1), strides=(4,4,1), padding='valid', activation='tanh'))
stae_model.compile(optimizer='adam', loss='mean_squared_error', metrics=['accuracy'])

/home/ubuntu24/PROJECTS/Intelligent-Video-Surveillance-Project/venv/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
E0000 00:00:1791321072.792121   95144 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


In [7]:
training_data = np.load('training.npy')
frames = training_data.shape[2]
frames = frames - (frames % 10)  # Ensure frames is a multiple of 10

In [8]:
training_data = np.transpose(training_data[:, :, :frames], (2, 0, 1))   # (N, H, W)
training_data = training_data.reshape(-1, 10, 227, 227)                 # (680, 10, H, W)
training_data = np.transpose(training_data, (0, 2, 3, 1))               # (680, H, W, 10)
training_data = np.expand_dims(training_data, axis=4)                   # (680, H, W, 10, 1)
target_data = training_data                                             # no copy, saves 1.4 GB

In [9]:
epochs = 5
batch_size = 1

In [10]:
callback_save = ModelCheckpoint("saved_model.keras", monitor="loss", save_best_only=True)
callback_early_stopping = EarlyStopping(monitor="loss", patience=3)

In [11]:
stae_model.fit(training_data, target_data, batch_size=batch_size, epochs=epochs, callbacks=[callback_save, callback_early_stopping])
stae_model.save("saved_model.keras")

W0000 00:00:1791321104.653430   95144 cpu_allocator_impl.cc:82] Allocation of 1401588800 exceeds 10% of free system memory.
W0000 00:00:1791321126.237848   95144 cpu_allocator_impl.cc:82] Allocation of 1401588800 exceeds 10% of free system memory.


Epoch 1/5


W0000 00:00:1791321138.089879   95627 cpu_allocator_impl.cc:82] Allocation of 86528000 exceeds 10% of free system memory.
W0000 00:00:1791321138.246700   95626 cpu_allocator_impl.cc:82] Allocation of 86528000 exceeds 10% of free system memory.
W0000 00:00:1791321138.281547   95629 cpu_allocator_impl.cc:82] Allocation of 21565440 exceeds 10% of free system memory.


680/680 ━━━━━━━━━━━━━━━━━━━━ 818s 1s/step - accuracy: 1.2785e-06 - loss: 0.0106
Epoch 2/5
680/680 ━━━━━━━━━━━━━━━━━━━━ 756s 1s/step - accuracy: 1.3727e-06 - loss: 0.0034
Epoch 3/5
680/680 ━━━━━━━━━━━━━━━━━━━━ 736s 1s/step - accuracy: 1.5440e-06 - loss: 0.0024
Epoch 4/5
680/680 ━━━━━━━━━━━━━━━━━━━━ 827s 1s/step - accuracy: 1.6410e-06 - loss: 0.0020
Epoch 5/5
680/680 ━━━━━━━━━━━━━━━━━━━━ 659s 968ms/step - accuracy: 1.7209e-06 - loss: 0.0017
